In [1]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import LatentDirichletAllocation
from bertopic import BERTopic
import matplotlib.pyplot as plt
import pyLDAvis
import pyLDAvis.lda_model
import numpy as np

In [2]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [3]:
df = pd.read_csv('BBC_data_CLEAN.csv')
df.head()

,Index,Text,Category
0,1830,Microsoft releases bumper patches Microsoft h...,tech
1,2181,What price for 'trusted PC security'? You can...,tech
2,2142,Why Cell will get the hard sell The world is ...,tech
3,1948,The Force is strong in Battlefront The warm r...,tech
4,1918,Mobile games come of age The BBC News website...,tech


In [4]:
num_clusters = df['Category'].nunique()
df['Category'].unique()

array(['tech', 'sport', 'politics', 'entertainment', 'business'],
      dtype=object)

In [5]:
df = pd.read_csv('BBC_data_CLEAN.csv')
text_column = 'Text'

df.dropna(subset=[text_column], inplace=True)
df.reset_index(drop=True, inplace=True)

corpus = df[text_column].tolist()
df.shape

(1735, 3)

In [6]:
lemmatizer = WordNetLemmatizer()
stop_words_nltk = set(stopwords.words('english'))

In [7]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words_nltk]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [8]:
processed_corpus = [preprocess_text(doc) for doc in corpus]

In [9]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english', max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_corpus)

count_vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english', max_features=1000)
count_matrix = count_vectorizer.fit_transform(processed_corpus)

In [10]:
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(tfidf_matrix)
silhouette_kmeans = silhouette_score(tfidf_matrix, kmeans_labels)
print(f"  K-Means Silhouette Score: {silhouette_kmeans:.4f}")

  K-Means Silhouette Score: 0.0306


In [11]:
agglomerative = AgglomerativeClustering(n_clusters=num_clusters)
agglomerative_labels = agglomerative.fit_predict(tfidf_matrix.toarray())
silhouette_hierarchical = silhouette_score(tfidf_matrix, agglomerative_labels)
print(f"  Hierarchical Clustering Silhouette Score: {silhouette_hierarchical:.4f}")

  Hierarchical Clustering Silhouette Score: 0.0214


In [12]:
lda = LatentDirichletAllocation(n_components=num_clusters, random_state=42, learning_method='online')
lda.fit(count_matrix)

LatentDirichletAllocation(learning_method='online', n_components=5,
                          random_state=42)

In [13]:
def display_topics(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Тема {topic_idx}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]))

print("Топ слов для каждой темы (LDA):")
display_topics(lda, count_vectorizer.get_feature_names_out(), 10)

Топ слов для каждой темы (LDA):
Тема 0:
said mr government people labour party election say tax tory
Тема 1:
said year film company award new market best firm sale
Тема 2:
said minister blair brown mr lord told eu european country
Тема 3:
said people music technology mobile phone service new year user
Тема 4:
game said time win player play year world england best


In [14]:
doc_topic_dist = lda.transform(count_matrix)

doc_lengths = np.asarray(count_matrix.sum(axis=1)).ravel()

term_frequency = np.asarray(count_matrix.sum(axis=0)).ravel()

vocab = count_vectorizer.get_feature_names_out()

topic_term_dists = lda.components_


lda_vis_data = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dist,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency,
    mds='tsne',
    sort_topics=False
)
pyLDAvis.save_html(lda_vis_data, 'lda_visualization.html')
print("\nLDA visualization saved to lda_visualization.html using pyLDAvis.prepare")


LDA visualization saved to lda_visualization.html using pyLDAvis.prepare


In [15]:
bertopic_model = BERTopic(verbose=True,
                          language="english",
                          nr_topics=num_clusters,
                          min_topic_size=10,
                          calculate_probabilities=True)

topics, probs = bertopic_model.fit_transform(corpus)

print(f"\nBERTopic обнаружил {len(bertopic_model.get_topic_info()) -1} тем (исключая тему выбросов -1).")

2025-05-26 19:52:41,798 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

2025-05-26 19:52:46,178 - BERTopic - Embedding - Completed ✓
2025-05-26 19:52:46,179 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-26 19:52:55,298 - BERTopic - Dimensionality - Completed ✓
2025-05-26 19:52:55,298 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-26 19:52:55,385 - BERTopic - Cluster - Completed ✓
2025-05-26 19:52:55,385 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-05-26 19:52:55,765 - BERTopic - Representation - Completed ✓
2025-05-26 19:52:55,765 - BERTopic - Topic reduction - Reducing number of topics
2025-05-26 19:52:55,777 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-26 19:52:56,197 - BERTopic - Representation - Completed ✓
2025-05-26 19:52:56,198 - BERTopic - Topic reduction - Reduced number of topics from 16 to 5



BERTopic обнаружил 4 тем (исключая тему выбросов -1).


In [16]:
print("BERTopic - Информация о темах:")
topic_info_df = bertopic_model.get_topic_info()
print(topic_info_df)

BERTopic - Информация о темах:
   Topic  Count              Name  \
0     -1     72  -1_the_to_of_and   
1      0    676    0_the_to_of_in   
2      1    353   1_the_to_in_and   
3      2    341   2_the_and_of_in   
4      3    293   3_the_to_of_and   

                                    Representation  \
0    [the, to, of, and, in, is, it, for, that, be]   
1  [the, to, of, in, and, said, that, for, on, is]   
2    [the, to, in, and, of, he, but, for, was, on]   
3   [the, and, of, in, to, for, on, was, film, is]   
4   [the, to, of, and, in, that, is, it, for, are]   

                                 Representative_Docs  
0  [Rivals of the 400 Apple...  The Mac mini is t...  
1  [Howard's unfinished business  "He's not finis...  
2  [White prepared for battle  Tough-scrummaging ...  
3  [Scissor Sisters triumph at Brits  US band Sci...  
4  [Who do you think you are?  The real danger is...  


In [17]:
print("\nBERTopic - Топ слова для тем:")
num_topics_to_display_bertopic = min(5, len(topic_info_df[topic_info_df.Topic != -1]))
for i in range(num_topics_to_display_bertopic):
    topic_words = bertopic_model.get_topic(i)
    print(f"Тема {i}: {', '.join([word[0] for word in topic_words[:10]])}")


BERTopic - Топ слова для тем:
Тема 0: the, to, of, in, and, said, that, for, on, is
Тема 1: the, to, in, and, of, he, but, for, was, on
Тема 2: the, and, of, in, to, for, on, was, film, is
Тема 3: the, to, of, and, in, that, is, it, for, are


In [18]:
df['BERTopic_Topic'] = topics

In [19]:
valid_indices = [i for i, topic in enumerate(topics) if topic != -1]
if len(valid_indices) > 0 and len(set(np.array(topics)[valid_indices])) > 1:
    bertopic_labels_valid = np.array(topics)[valid_indices]
    tfidf_matrix_valid_for_bertopic = tfidf_matrix[valid_indices]
    silhouette_bertopic = silhouette_score(tfidf_matrix_valid_for_bertopic, bertopic_labels_valid)
    print(f"\n  BERTopic Silhouette Score (на TF-IDF, без выбросов): {silhouette_bertopic:.4f}")



  BERTopic Silhouette Score (на TF-IDF, без выбросов): 0.0275


In [ ]:
 fig_topics = bertopic_model.visualize_topics()
 fig_topics.show()
 fig_hierarchy = bertopic_model.visualize_hierarchy()
 fig_hierarchy.show()
 fig_barchart = bertopic_model.visualize_barchart(top_n_topics=num_clusters if 'num_clusters' in locals() else 5)
 fig_barchart.show()

In [21]:
pyLDAvis.display(lda_vis_data)